Run the following code block to initialize most of the helper functions

In [1]:
from collections import defaultdict
from openai import OpenAI
import pandas as pd
import os
import json
import psycopg2
import openai
from bs4 import BeautifulSoup
import re

openaiClient = openai.OpenAI(
    api_key= os.getenv("OPENAI_API_KEY"),
    organization= os.getenv("OPENAPI_ORG")
    )

PgVectorConn = psycopg2.connect(
    host=os.getenv("POSTGRES_HOST"),
    database=os.getenv("POSTGRES_DATABASE"),
    user=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD"),
    port=os.getenv("POSTGRES_PORT")
)

In [2]:
import uuid
from pydantic import BaseModel, Field
from typing import TypedDict, List, Optional
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage

class HittechLineItem(BaseModel):
    """
    A line item that contains the general information for a bought/sold unit.'
    """
    quantity: Optional[str] = Field(
        ...,
        description="De ID of het nummer van het genoemde wetsartikel (bijv. 'Art. 5' of '53475')."
    )
    unit: Optional[str] = Field(
        ...,
        description="De ID of het nummer van het genoemde wetsartikel (bijv. 'Art. 5' of '53475')."
    )
    item: Optional[str] = Field(
        ...,
        description="Name of the actual item, part of a paragraph with other information such as PO nnumber, line number and item-number."
    )
    target_shipping_date: Optional[str] = Field(
        ...,
        description="Targetted shipping date, in the format of yyyy-MMM-dd"
    )
    price_unit: Optional[str] = Field(
        ...,
        description="Price per unit for this line item."
    )
    total: Optional[str] = Field(
        ...,
        description="The total price of all of the units."
    )

class HittechHeader(BaseModel):
    """
    Header information of the current order acknowledgement.
    """
    order_date: Optional[str] = Field(
        ...,
        description="Date when the order was admitted."
    )
    customer_number: Optional[str] = Field(
        ...,
        description="Customer order number."
    )
    order_number: Optional[str] = Field(
        ...,
        description="Your order number."
    )
    order_issued: Optional[str] = Field(
        ...,
        description="Date when the order was issued."
    )
    delivery_address: Optional[str] = Field(
        ...,
        description="Address for delivery. Usually grouped with the target company."
    )

class HittechDoc(BaseModel):
    """
    Een gestructureerde weergave van een wettekst, inclusief verwijzingen
    naar andere wetsartikelen of codes.
    """
    lineitems: List[HittechLineItem] = Field(
        default_factory=list,
        description="A list of line items from an order acknowledgement."
    )
    header: Optional[HittechHeader] = Field(
        ...,
        description="Information from the header."
    )

class Example(TypedDict):
    input: str
    tool_calls: List[BaseModel]

def tool_example_to_messages(example: Example) -> List[BaseMessage]:
    """Convert an example into a list of messages that can be fed into a language model."""
    messages: List[BaseMessage] = [HumanMessage(content=example["input"])]
    tool_calls = []
    
    for tool_call in example["tool_calls"]:
        tool_calls.append(
            {
                "id": str(uuid.uuid4()),
                "args": tool_call.dict(),
                "name": tool_call.__class__.__name__,
            },
        )
    
    messages.append(AIMessage(content="", tool_calls=tool_calls))
    
    tool_outputs = example.get("tool_outputs") or [
        "You have correctly called this tool."
    ] * len(tool_calls)
    
    for output, tool_call in zip(tool_outputs, tool_calls):
        messages.append(ToolMessage(content=output, tool_call_id=tool_call["id"]))
    
    return messages


def create_examples_and_messages() -> List[BaseMessage]:
    """
    Builds an example set of messages demonstrating how a hittech
    document can be turned into a structured object.
    """
    
    examples = [
        (
            "Volgens het Vlaamse Milieuwetboek (editie 2025) bepaalt artikel 53475 de reikwijdte van afvalbeheer. Daarnaast verduidelijkt Art. 12bis de beroepsprocedure. Deze tekst is gepubliceerd op 2025-02-10.",
            HittechDoc(
                lineitems=[
                    HittechLineItem(
                        quantity="10",
                        unit="pcs",
                        item="Widget",
                        target_shipping_date="2025-Mar-15",
                        price_unit="2.50 EUR",
                        total="25.00 EUR"
                    ),
                    HittechLineItem(
                        quantity="5",
                        unit="pcs",
                        item="Gadget",
                        target_shipping_date="2025-Mar-20",
                        price_unit="5.00 EUR",
                        total="25.00 EUR"
                    ),
                ],
                header=HittechHeader(
                    order_date="2025-02-10",
                    customer_number="98765",
                    order_number="ABC123",
                    order_issued="2025-01-31",
                    delivery_address="Some Street 123, Flanders"
                )
            )
        )
    ]

    messages = []

    for text, tool_call in examples:
        messages.extend(
            tool_example_to_messages(
                {"input": text, "tool_calls": [tool_call]}
            )
        )

    return messages

In [3]:
import os
from PyPDF2 import PdfReader
from langchain.chat_models import init_chat_model
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.callbacks import get_openai_callback

# Instantiate your language model (replace with your own configuration)
llm = init_chat_model(
    model="gpt-4o-mini",
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY"),
    organization= os.getenv("OPENAPI_ORG")
    )

def process_file(filepath: str):
    """
    Processes a machine-readable PDF and returns extracted data in a structured format.
    """

    # Build your system + examples + human prompt
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """You are an expert extraction algorithm.
                Only extract relevant information from the text.
                Only extract data found from the pdf, so never take data only found from the examples given.
                The ship_to and sold_to fields you are retrieving can never be related to ArcelorMittal.
                If you do not know the value of an attribute asked
                to extract, return null for the attribute's value.
                If there are multiple line items, you will return all of them as an array under the LineItems field.
                The amount of LineItems can be determined using the LineItem_Number.
                Below is an example of the JSON format to return.
                If there is only ONE line item, 'LineItems' contains only that one object.
                Example items that are linked to similar items are given below.\n""",
            ),
            # Use the placeholder for examples
            MessagesPlaceholder("examples"),
            # Finally, the human prompt to feed in the PDF text
            ("human", "{text}"),
        ]
    )

    # Connect your prompt to an LLM with structured output
    runnable = prompt | llm.with_structured_output(
        schema=HittechDoc,
        method="function_calling",
        include_raw=False
    )

    # --- Use PyPDF to extract text from a machine-readable PDF ---
    text_content = ""
    with open(filepath, "rb") as f:
        pdf = PdfReader(f)
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text_content += page_text + "\n"
    # -------------------------------------------------------------

    # Build example messages (from your code)
    messages = create_examples_and_messages()

    # Run the chain with PDF text as "text" and example messages as "examples"
    with get_openai_callback() as cb:
        processed_data = runnable.invoke(
            {
                "text": text_content,
                "examples": messages
            }
        )

    return processed_data, cb

In [10]:
test_data = process_file("4021046682.pdf")

In [11]:
test_data

(HittechDoc(lineitems=[HittechLineItem(quantity='6', unit='st', item='EPE-F/45/Black/200/200/100 XPE 45kg/m3 foambufferset. 200x200x100mm. Acc. to drawing 3005427-A', target_shipping_date='2025-01-17', price_unit='50.00 EUR', total='300.00 EUR')], header=HittechHeader(order_date='2025-01-17', customer_number='100727', order_number='4021046682', order_issued='2025-01-23', delivery_address='LAAN VAN YPENBURG 62, 2497 GB DEN HAAG')),
 Tokens Used: 1466
 	Prompt Tokens: 1322
 		Prompt Tokens Cached: 0
 	Completion Tokens: 144
 		Reasoning Tokens: 0
 Successful Requests: 1
 Total Cost (USD): $0.0002847)

In [12]:
for lineitem in test_data[0].lineitems:
    print(lineitem)

for i in test_data[0].header:
    print(i)

print(test_data[1])

quantity='6' unit='st' item='EPE-F/45/Black/200/200/100 XPE 45kg/m3 foambufferset. 200x200x100mm. Acc. to drawing 3005427-A' target_shipping_date='2025-01-17' price_unit='50.00 EUR' total='300.00 EUR'
('order_date', '2025-01-17')
('customer_number', '100727')
('order_number', '4021046682')
('order_issued', '2025-01-23')
('delivery_address', 'LAAN VAN YPENBURG 62, 2497 GB DEN HAAG')
Tokens Used: 1466
	Prompt Tokens: 1322
		Prompt Tokens Cached: 0
	Completion Tokens: 144
		Reasoning Tokens: 0
Successful Requests: 1
Total Cost (USD): $0.0002847


Run the following code block to get unique items ready for testing. These items have not yet been placed in the database (We keep a list of all items in the database too in the Article_Objects folder)

In [5]:
folders_to_check = ["data/Stoffen", "data/Handhaving", "data/Compartiment"]
comparison_folder = "Article_Objects"

# Get the unique articles
unique_articles = get_unique_articles_from_folders(folders_to_check, comparison_folder)

In [10]:
from vito_utils.utils import clean_html
clean_html(unique_articles[0].inhoud)

'Elke natuurlijke persoon of rechtspersoon die gevestigd is op het grondgebied en die, via verkoop op afstand, rechtstreeks of door gebruik van een onlinemarktplaats EEA verkoopt aan particuliere huishoudens of aan andere gebruikers dan particuliere huishoudens buiten het grondgebied, wijst binnen dat grondgebied een natuurlijke persoon of rechtspersoon aan als de gevolmachtigde die verantwoordelijk is voor het nakomen van de verplichtingen als producent van EEA, die uit de wetgeving van dat land met betrekking tot de uitgebreide producentenverantwoordelijkheid voortvloeien. Elke natuurlijke persoon of rechtspersoon die gevestigd is buiten het grondgebied en die, via verkoop op afstand, rechtstreeks EEA verkoopt aan particuliere huishoudens of aan andere gebruikers dan particuliere huishoudens op het grondgebied, wijst een op het grondgebied gevestigde natuurlijke persoon of rechtspersoon aan als gevolmachtigde die verantwoordelijk is voor het nakomen van de verplichtingen van de produ

In [1]:
from vito_utils.loading import get_unique_articles_from_folders
from vito_utils.vito_classes import VitoArticle
test_item = VitoArticle('C:/Users/EhranLenaerts/Documents/RAG_B_Robots/RAG_Vito/data/Stoffen/Aard/Afvalstoffen/Bedrijfsafval/glasafval/53475.json')

c:\Users\EhranLenaerts\anaconda3\envs\VITO\lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


In [2]:
from vito_utils.utils import clean_html
test_item.inhoud = clean_html(test_item.inhoud)

In [3]:
test_item.inhoud

'Product Specificatie Stuifcategorie Voetnoot Abbrände pyrietas SC2 Aluinaarde SC1 Bariet SC1 Bariet gemalen SC1 Bauxiet China gecalcineerd SC1 gecalcineerd SC1 ruw bauxiet SC3 Bimskies SC2 Borax SC1 Bodemas vochtgehalte 30  SC2 4 Bruinsteen SC2 Calcium Carbid SC1 Carborundum SC3 Cement vochtgehalte 0,3  SC1 5 Klinker grondstof SC1 Cokes steenkoolcokes SC2 petroleumcokes, grof SC2 petroleumcokes, fijn SC2 petroleumcokes, gecalcineerd SC1 petroleumcokes oilednon-oiled SC2 5 fluid cokes SC1 Derivaten en aanverwante producten aardappelmeel SC1 aardappelschijfjes SC1 alfalfapellets SC1 amandelmeel SC1 appelpulppellets SC1 babassupellets SC1 babassuschroot SC1 beendermeel SC1 beenderschroot SC1 bierbostelpellets SC1 bladmeelpellets SC1 boekweitmeel SC1 cacaobonen SC1 3 corndistillergrainpellets SC1 corndistillergrainmeel SC1 corncobpellets SC1 cornplantpellets SC1 citruspellets SC1 D.F.G. pellets maiskiempellets SC1 druivenpulpgranulaat SC2 5 gerstemeel SC1 gerstpellets SC1 grondnoten SC3 g

In [10]:
test_item.extract_keywords()

In [11]:
test_item.keywords

test_item.keywords[0]

('steenkoolcokes sc2', 0.6353)

In [12]:
test_item.weighted_keywords

[{'steenkoolcokes sc2': 0.6353},
 {'gemalen sc1 bauxiet': 0.6229},
 {'mengvoederpellets sc1 millrunpellets': 0.6131},
 {'sc3 pyrietas sc2': 0.6119},
 {'sc1 miloglutenpellets': 0.6089},
 {'sojaschroot sc1 splentgrainpellets': 0.6077},
 {'sc1 citruspellets': 0.6065},
 {'sc1 corndistillergrainpellets': 0.6043},
 {'katoenzaadpellets sc1 katoenzaadschroot': 0.6041},
 {'cokes steenkoolcokes sc2': 0.6035}]

In [11]:
from vito_utils.keyword_utils import extract_keybert_keywords_local

result = extract_keybert_keywords_local(unique_articles[0].inhoud)

To get random testing items from the unique items, run the following code block

In [142]:
import random
random_unique_articles = random.sample(unique_articles, 50)

In [ ]:
articles_raw = get_all_items(PgVectorConn)
articles = []
for raw_article in articles_raw:
    articles.append({
        'id': raw_article[0],
        'keywords': raw_article[2],
        'labels': parse_labels(raw_article[3])
    })

In [ ]:
# Building Keyword sets to remove randomness
from collections import defaultdict

keyword_label_counts = defaultdict(lambda: defaultdict(int))
keyword_total_counts = defaultdict(int)
total_articles = len(articles)

# Build counts
for article in articles:
    keywords = article['keywords']
    labels = article['labels']
    unique_keywords = set(keywords)
    unique_labels = set(labels)
    for keyword in unique_keywords:
        keyword_total_counts[keyword] += 1
        for label in unique_labels:
            keyword_label_counts[keyword][label] += 1

# Calculate normalized frequencies
keyword_label_assoc = defaultdict(dict)
for keyword, label_counts in keyword_label_counts.items():
    total_keyword_count = keyword_total_counts[keyword]
    for label, count in label_counts.items():
        # Normalized frequency: P(Label | Keyword)
        normalized_freq = count / total_keyword_count
        keyword_label_assoc[keyword][label] = normalized_freq

min_frequency = 5 

# Identify and remove rare keywords
for article in articles:
    article['keywords'] = [
        keyword for keyword in article['keywords']
        if keyword_total_counts[keyword] >= min_frequency
    ]
for article in articles:
    article_labels = set(article['labels'])
    article_keywords = article['keywords']
    weighted_keywords = {}
    for keyword in article_keywords:
        # Get association scores of the keyword with the article's labels
        label_assoc_scores = [
            keyword_label_assoc[keyword].get(label, 0) for label in article_labels
        ]
        if label_assoc_scores:
            # Use the average association score
            keyword_weight = sum(label_assoc_scores) / len(label_assoc_scores)
            weighted_keywords[keyword] = keyword_weight
        else:
            # If no association, weight can be set to zero or a minimal value
            weighted_keywords[keyword] = 0
    # Update the article with weighted keywords
    article['weighted_keywords'] = weighted_keywords

for article in articles:
    weights = list(article['weighted_keywords'].values())
    total_weight = sum(weights)
    if total_weight > 0:
        article['weighted_keywords'] = {
            keyword: weight / total_weight
            for keyword, weight in article['weighted_keywords'].items()
            }

In [ ]:
cursor = PgVectorConn.cursor()

for article in articles:
    article_id = article['id']
    weighted_keywords = article['weighted_keywords']
    weighted_keywords_json = json.dumps(weighted_keywords)
    cursor.execute("""
        UPDATE vito_article
        SET weighted_keywords = %s
        WHERE artikel_id = %s
    """, (weighted_keywords_json, article_id))

PgVectorConn.commit()
cursor.close()
PgVectorConn.close()

Run the following code block to instantiate the A* functionality

Example of processing a single item

In [361]:
List_result_labels = []
article = random_unique_articles[26]
if article.inhoud == "<div>\r\n     <div>[...]</div>\r\n   </div>":
    print('----------------------------------------------------------------------------------------------------------------')
    print("Skipping due to invalid HTML/article")
else:
    # try:
        article.embedding = article.get_embedding(openaiClient)
        keyword_result = UploadOpenAI(article.inhoud, openaiClient)
        # Adding keywords in a list format
        article.add_keywords([keyword['Keyword'] for keyword in json.loads(keyword_result.choices[0].message.content)["metadata"]])
        result_labels, assigned_labels = assign_labels_a_star(article)
        List_result_labels.append({
            "Artikel": article.artikel_id, 
            "Artikel_name": article.artikel, 
            "List_result_labels": result_labels, 
            "correct_labels": article.labels})
        print('----------------------------------------------------------------------------------------------------------------')
        print(f"Article: {article.artikel_id}, Labels: {result_labels}, correct labels: {article.labels}")
    # except Exception as e:
    #     print(f"Article: {article.artikel_id}, {e}")

---------------------------------------------------------------------------

Stage 0, Level: Initial items

Article ID: 79326
Labels: ['Stoffen', 'Aard', 'Chemische stoffen', 'Anorganische verbindingen', 'Andere', 'stof']
Semantic similarity: 2.871468987131477
Combined score: 2.871468987131477

Test Article Keywords: ['geleide emissies', 'zwaveldioxide', 'zure gassen', 'verlaagd', 'natte wassing']
Article Keywords: ['lucht', 'technieken', 'procesovens', 'BBT-conclusies', 'organisch-chemische producten']
No overlapping keywords.

Article ID: 79327
Labels: ['Stoffen', 'Aard', 'Chemische stoffen', 'Anorganische verbindingen', 'Specifieke anorganische verbindingen', 'zwaveloxiden']
Semantic similarity: 2.861315731992804
Combined score: 2.861315731992804

Test Article Keywords: ['geleide emissies', 'zwaveldioxide', 'zure gassen', 'verlaagd', 'natte wassing']
Article Keywords: ['technieken', 'toepassing', 'procesovens', 'BBT-conclusies', 'organisch-chemische producten']
No overlapping keywor

In [138]:
List_result_labels = []
for article in random_unique_articles[:25]:
    if article.inhoud == "<div>\r\n     <div>[...]</div>\r\n   </div>":
        print("\nSkipping, incorrect HTML/article")
        continue
    else:
        try:
            article.embedding = article.get_embedding(openaiClient)
            keyword_result = UploadOpenAI(article.inhoud, openaiClient)
            article.add_keywords([keyword['Keyword'] for keyword in json.loads(keyword_result.choices[0].message.content)["metadata"]])
            result_labels, assigned_labels = assign_labels_a_star(article)
            List_result_labels.append({
                "Artikel": article.artikel_id,
                "Artikel_name": article.artikel,
                "List_result_labels": result_labels,
                "correct_labels": article.labels,
                "score_chosen_items": assigned_labels[0]['mean_score']})
            print('----------------------------------------------------------------------------------------------------------------')
            print(f"Article: {article.artikel_id}, Labels: {result_labels}, correct labels: {article.labels}")
        except Exception as e:
            print(f"Article: {article.artikel_id}, {e}")

---------------------------------------------------------------------------

Stage 0, Level: Initial items

Article ID: 60482
Labels: ['Stoffen', 'Aard', 'Afvalstoffen', 'Bijzondere afvalstoffen', 'dierlijk afval']
Semantic similarity: 2.737980424827296
Combined score: 2.737980424827296

Test Article Keywords: ['bevoegde autoriteit', 'illegale behandeling', 'huiden van dieren', 'Richtlijn 96/22/EG', 'TSE besmetting', 'categorie 1-materiaal', 'vervoedering', 'cosmetische producten', 'diergeneeskundige geneesmiddelen', 'gezondheidscertificering']
Article Keywords: []
No overlapping keywords.

Article ID: 60482
Labels: ['Stoffen', 'Aard', 'Afvalstoffen', 'Bijzondere afvalstoffen', 'dierlijk afval']
Semantic similarity: 2.7322318661245144
Combined score: 2.7322318661245144

Test Article Keywords: ['bevoegde autoriteit', 'illegale behandeling', 'huiden van dieren', 'Richtlijn 96/22/EG', 'TSE besmetting', 'categorie 1-materiaal', 'vervoedering', 'cosmetische producten', 'diergeneeskundige ge

In [139]:
# Calculate accuracies and median scores
detailed_results, top1_accuracy, topn_accuracy, median_top1_score, median_topn_score = calculate_label_accuracy_per_level(List_result_labels)

# Print Top-1 Accuracy per Label Level
print("Top-1 Accuracy per Label Level:")
for level, accuracy in top1_accuracy.items():
    print(f"Level {level}: {accuracy:.2f}%")
print(f"Median Top-1 Score: {median_top1_score:.4f}")

# Print Top-N Accuracy per Label Level
print("\nTop-N Accuracy per Label Level:")
for level, accuracy in topn_accuracy.items():
    print(f"Level {level}: {accuracy:.2f}%")
print(f"Median Top-N Score: {median_topn_score:.4f}")

Top-1 Accuracy per Label Level:
Level 0: 81.82%
Level 1: 72.73%
Level 2: 72.73%
Median Top-1 Score: 5.2249

Top-N Accuracy per Label Level:
Level 0: 81.82%
Level 1: 72.73%
Level 2: 72.73%
Median Top-N Score: 5.2249


In [143]:
List_result_labels = []
for article in random_unique_articles[:25]:
    if article.inhoud == "<div>\r\n     <div>[...]</div>\r\n   </div>":
        print("\nSkipping, incorrect HTML/article")
        continue
    else:
        try:
            article.embedding = article.get_embedding(openaiClient)
            keyword_result = UploadOpenAI(article.inhoud, openaiClient)
            article.add_keywords([keyword['Keyword'] for keyword in json.loads(keyword_result.choices[0].message.content)["metadata"]])
            result_labels, assigned_labels = assign_labels_a_star(article)
            List_result_labels.append({
                "Artikel": article.artikel_id,
                "Artikel_name": article.artikel,
                "List_result_labels": result_labels,
                "correct_labels": article.labels,
                "score_chosen_items": assigned_labels[0]['mean_score']})
            print('----------------------------------------------------------------------------------------------------------------')
            print(f"Article: {article.artikel_id}, Labels: {result_labels}, correct labels: {article.labels}")
        except Exception as e:
            print(f"Article: {article.artikel_id}, {e}")

---------------------------------------------------------------------------

Stage 0, Level: Initial items

Article ID: 77364
Labels: ['Stoffen', 'Aard', 'Afvalstoffen', 'Gevaarlijke afvalstoffen', 'gevaarlijke afvalstoffen']
Semantic similarity: 4.042137482765905
Combined score: 5.042137482765905

Test Article Keywords: ['asbesthoudende materialen', 'zonnepanelen', 'reclamepanelen', 'asbestveilige toestand', 'risicobeheersmaatregel', 'veilig beheer', 'dakbekleding', 'gevelbekleding', 'insluiten', 'bedekken']
Article Keywords: ['asbest']
Overlapping Keywords:
Test Keyword: 'asbesthoudende materialen', Article Keyword: 'asbest', Article Keyword Weight: 1.0
Test Keyword: 'asbestveilige toestand', Article Keyword: 'asbest', Article Keyword Weight: 1.0

Article ID: 77379
Labels: ['Stoffen', 'Aard', 'Afvalstoffen', 'Bedrijfsafval', 'asbesthoudende afvalstoffen']
Semantic similarity: 2.750415006999438
Combined score: 2.750415006999438

Test Article Keywords: ['asbesthoudende materialen', 'zo

In [144]:
# Calculate accuracies and median scores
detailed_results, top1_accuracy, topn_accuracy, median_top1_score, median_topn_score = calculate_label_accuracy_per_level(List_result_labels)

# Print Top-1 Accuracy per Label Level
print("Top-1 Accuracy per Label Level:")
for level, accuracy in top1_accuracy.items():
    print(f"Level {level}: {accuracy:.2f}%")
print(f"Median Top-1 Score: {median_top1_score:.4f}")

# Print Top-N Accuracy per Label Level
print("\nTop-N Accuracy per Label Level:")
for level, accuracy in topn_accuracy.items():
    print(f"Level {level}: {accuracy:.2f}%")
print(f"Median Top-N Score: {median_topn_score:.4f}")

Top-1 Accuracy per Label Level:
Level 0: 90.00%
Level 1: 80.00%
Level 2: 80.00%
Median Top-1 Score: 4.3998

Top-N Accuracy per Label Level:
Level 0: 90.00%
Level 1: 80.00%
Level 2: 80.00%
Median Top-N Score: 4.3998


In [133]:
List_result_labels = []
for article in random_unique_articles[25:]:
    if article.inhoud == "<div>\r\n     <div>[...]</div>\r\n   </div>":
        print("\nSkipping, incorrect HTML/article")
        continue
    else:
        try:
            article.embedding = article.get_embedding(openaiClient)
            keyword_result = UploadOpenAI(article.inhoud, openaiClient)
            article.add_keywords([keyword['Keyword'] for keyword in json.loads(keyword_result.choices[0].message.content)["metadata"]])
            result_labels, assigned_labels = assign_labels_a_star(article)
            List_result_labels.append({
                "Artikel": article.artikel_id,
                "Artikel_name": article.artikel,
                "List_result_labels": result_labels,
                "correct_labels": article.labels,
                "score_chosen_items": assigned_labels[0]['mean_score']})
            print('----------------------------------------------------------------------------------------------------------------')
            print(f"Article: {article.artikel_id}, Labels: {result_labels}, correct labels: {article.labels}")
        except Exception as e:
            print(f"Article: {article.artikel_id}, {e}")

---------------------------------------------------------------------------

Stage 0, Level: Initial items

Article ID: 79382
Labels: ['Stoffen', 'Aard', 'Chemische stoffen', 'Organische verbindingen', 'Niet-gehalogeneerde verbindingen', 'MAK']
Semantic similarity: 3.477181356722891
Combined score: 3.5533824479053195

Test Article Keywords: ['processpecifieke bepalingen', 'ethylbenzeen', 'zeoliet', 'AlCl3', 'gekatalyseerd alkyleringsproces', 'styreenmonomeer', 'dehydrogenering', 'coproductie', 'propyleenoxide', 'productieproces']
Article Keywords: ['afval', 'benzeen', 'emissies', 'afvalwater', 'zure gassen', 'installaties', 'organische verbindingen']
Overlapping Keywords:
Test Keyword: 'ethylbenzeen', Article Keyword: 'benzeen', Article Keyword Weight: 0.1524021823648573

Article ID: 79396
Labels: ['Stoffen', 'Aard', 'Chemische stoffen', 'Organische verbindingen', 'Niet-gehalogeneerde verbindingen', 'MAK']
Semantic similarity: 2.7667576146459014
Combined score: 2.7667576146459014

Test

In [136]:
# Calculate accuracies and median scores
detailed_results, top1_accuracy, topn_accuracy, median_top1_score, median_topn_score = calculate_label_accuracy_per_level(List_result_labels)

# Print Top-1 Accuracy per Label Level
print("Top-1 Accuracy per Label Level:")
for level, accuracy in top1_accuracy.items():
    print(f"Level {level}: {accuracy:.2f}%")
print(f"Median Top-1 Score: {median_top1_score:.4f}")

# Print Top-N Accuracy per Label Level
print("\nTop-N Accuracy per Label Level:")
for level, accuracy in topn_accuracy.items():
    print(f"Level {level}: {accuracy:.2f}%")
print(f"Median Top-N Score: {median_topn_score:.4f}")

Top-1 Accuracy per Label Level:
Level 0: 76.19%
Level 1: 71.43%
Level 2: 61.90%
Median Top-1 Score: 3.3714

Top-N Accuracy per Label Level:
Level 0: 76.19%
Level 1: 71.43%
Level 2: 61.90%
Median Top-N Score: 3.3714


Runn the following to process batch of items that were previously randomly selected

In [ ]:
List_result_labels = []
for article in random_unique_articles:
    if article.inhoud == "<div>\r\n     <div>[...]</div>\r\n   </div>":
        print("\nSkipping, incorrect HTML/article")
        continue
    else:
        try:
            article.embedding = article.get_embedding(openaiClient)
            keyword_result = UploadOpenAI(article.inhoud, openaiClient)
            article.add_keywords([keyword['Keyword'] for keyword in json.loads(keyword_result.choices[0].message.content)["metadata"]])
            result_labels, assigned_labels = assign_labels_a_star(article)
            List_result_labels.append({
                "Artikel": article.artikel_id,
                "Artikel_name": article.artikel,
                "List_result_labels": result_labels,
                "correct_labels": article.labels,
                "score_chosen_items": assigned_labels[0]['mean_score']})
            print('----------------------------------------------------------------------------------------------------------------')
            print(f"Article: {article.artikel_id}, Labels: {result_labels}, correct labels: {article.labels}")
        except Exception as e:
            print(f"Article: {article.artikel_id}, {e}")

Run the following code block to generalize scores for a better overview

In [119]:
# Calculate accuracies and median scores
detailed_results, top1_accuracy, topn_accuracy, median_top1_score, median_topn_score = calculate_label_accuracy_per_level(List_result_labels)

# Print Top-1 Accuracy per Label Level
print("Top-1 Accuracy per Label Level:")
for level, accuracy in top1_accuracy.items():
    print(f"Level {level}: {accuracy:.2f}%")
print(f"Median Top-1 Score: {median_top1_score:.4f}")

# Print Top-N Accuracy per Label Level
print("\nTop-N Accuracy per Label Level:")
for level, accuracy in topn_accuracy.items():
    print(f"Level {level}: {accuracy:.2f}%")
print(f"Median Top-N Score: {median_topn_score:.4f}")

Top-1 Accuracy per Label Level:
Level 0: 90.91%
Level 1: 79.55%
Level 2: 59.09%
Median Top-1 Score: 3.2584

Top-N Accuracy per Label Level:
Level 0: 90.91%
Level 1: 79.55%
Level 2: 70.45%
Median Top-N Score: 2.9578


In [121]:
List_result_labels = []
for article in random_unique_articles:
    if article.inhoud == "<div>\r\n     <div>[...]</div>\r\n   </div>":
        print("\nSkipping, incorrect HTML/article")
        continue
    else:
        try:
            article.embedding = article.get_embedding(openaiClient)
            keyword_result = UploadOpenAI(article.inhoud, openaiClient)
            article.add_keywords([keyword['Keyword'] for keyword in json.loads(keyword_result.choices[0].message.content)["metadata"]])
            result_labels, assigned_labels = assign_labels_a_star(article)
            List_result_labels.append({
                "Artikel": article.artikel_id,
                "Artikel_name": article.artikel,
                "List_result_labels": result_labels,
                "correct_labels": article.labels,
                "score_chosen_items": assigned_labels[0]['mean_score']})
            print('----------------------------------------------------------------------------------------------------------------')
            print(f"Article: {article.artikel_id}, Labels: {result_labels}, correct labels: {article.labels}")
        except Exception as e:
            print(f"Article: {article.artikel_id}, {e}")

---------------------------------------------------------------------------

Stage 0, Level: Initial items

Article ID: 39762
Labels: ['Compartiment', 'Energie']
Semantic similarity: 2.9192075131627404
Combined score: 3.12460170403411

Test Article Keywords: ['VREG', 'rechtspersoon', 'beheer', 'elektriciteitsdistributienet', 'aardgasdistributienet', 'geografisch afgebakend gebied', 'aangesloten afnemers', 'gemeente Voeren', 'gemeente Baarle-Hertog', 'netbeheerder']
Article Keywords: ['VREG', 'veiligheid', 'voorwaarden', 'toegangshouder']
Overlapping Keywords:
Test Keyword: 'VREG', Article Keyword: 'VREG', Article Keyword Weight: 0.4107883817427386

Article ID: 39762
Labels: ['Compartiment', 'Energie']
Semantic similarity: 2.9178049449182395
Combined score: 3.123199135789609

Test Article Keywords: ['VREG', 'rechtspersoon', 'beheer', 'elektriciteitsdistributienet', 'aardgasdistributienet', 'geografisch afgebakend gebied', 'aangesloten afnemers', 'gemeente Voeren', 'gemeente Baarle-Herto

In [122]:
# Calculate accuracies and median scores
detailed_results, top1_accuracy, topn_accuracy, median_top1_score, median_topn_score = calculate_label_accuracy_per_level(List_result_labels)

# Print Top-1 Accuracy per Label Level
print("Top-1 Accuracy per Label Level:")
for level, accuracy in top1_accuracy.items():
    print(f"Level {level}: {accuracy:.2f}%")
print(f"Median Top-1 Score: {median_top1_score:.4f}")

# Print Top-N Accuracy per Label Level
print("\nTop-N Accuracy per Label Level:")
for level, accuracy in topn_accuracy.items():
    print(f"Level {level}: {accuracy:.2f}%")
print(f"Median Top-N Score: {median_topn_score:.4f}")

Top-1 Accuracy per Label Level:
Level 0: 76.09%
Level 1: 67.39%
Level 2: 52.17%
Median Top-1 Score: 69.4197

Top-N Accuracy per Label Level:
Level 0: 76.09%
Level 1: 67.39%
Level 2: 60.87%
Median Top-N Score: 59.7122


In [125]:
List_result_labels = []
for article in random_unique_articles:
    if article.inhoud == "<div>\r\n     <div>[...]</div>\r\n   </div>":
        print("\nSkipping, incorrect HTML/article")
        continue
    else:
        try:
            article.embedding = article.get_embedding(openaiClient)
            keyword_result = UploadOpenAI(article.inhoud, openaiClient)
            article.add_keywords([keyword['Keyword'] for keyword in json.loads(keyword_result.choices[0].message.content)["metadata"]])
            result_labels, assigned_labels = assign_labels_a_star(article)
            List_result_labels.append({
                "Artikel": article.artikel_id,
                "Artikel_name": article.artikel,
                "List_result_labels": result_labels,
                "correct_labels": article.labels,
                "score_chosen_items": assigned_labels[0]['mean_score']})
            print('----------------------------------------------------------------------------------------------------------------')
            print(f"Article: {article.artikel_id}, Labels: {result_labels}, correct labels: {article.labels}")
        except Exception as e:
            print(f"Article: {article.artikel_id}, {e}")

---------------------------------------------------------------------------

Stage 0, Level: Initial items

Article ID: 44834
Labels: ['Handhaving', 'Toezicht', 'Modaliteiten', 'Recht van onderzoek van zaken']
Semantic similarity: 3.5489528945466553
Combined score: 3.9342739954640864

Test Article Keywords: ['bevoegde instantie', 'Commissie', 'criteria', 'informatievoorschriften', 'kennisgeving', 'handel', "GGO's", 'product', 'milieu', 'wetenschappelijk bewijs']
Article Keywords: ['gezondheid', 'kennisgeving']
Overlapping Keywords:
Test Keyword: 'kennisgeving', Article Keyword: 'kennisgeving', Article Keyword Weight: 0.7706422018348624

Article ID: 44848
Labels: ['Compartiment', 'Bodem']
Semantic similarity: 3.3229927805869033
Combined score: 3.8229927805869033

Test Article Keywords: ['bevoegde instantie', 'Commissie', 'criteria', 'informatievoorschriften', 'kennisgeving', 'handel', "GGO's", 'product', 'milieu', 'wetenschappelijk bewijs']
Article Keywords: ['gedelegeerde handelingen']

In [126]:
# Calculate accuracies and median scores
detailed_results, top1_accuracy, topn_accuracy, median_top1_score, median_topn_score = calculate_label_accuracy_per_level(List_result_labels)

# Print Top-1 Accuracy per Label Level
print("Top-1 Accuracy per Label Level:")
for level, accuracy in top1_accuracy.items():
    print(f"Level {level}: {accuracy:.2f}%")
print(f"Median Top-1 Score: {median_top1_score:.4f}")

# Print Top-N Accuracy per Label Level
print("\nTop-N Accuracy per Label Level:")
for level, accuracy in topn_accuracy.items():
    print(f"Level {level}: {accuracy:.2f}%")
print(f"Median Top-N Score: {median_topn_score:.4f}")

Top-1 Accuracy per Label Level:
Level 0: 71.79%
Level 1: 61.54%
Level 2: 46.15%
Median Top-1 Score: 2.3293

Top-N Accuracy per Label Level:
Level 0: 71.79%
Level 1: 61.54%
Level 2: 56.41%
Median Top-N Score: 2.1796


In [383]:
List_result_labels = []
for article in random_unique_articles:
    if article.inhoud == "<div>\r\n     <div>[...]</div>\r\n   </div>":
        print("\nSkipping, incorrect HTML/article")
        continue
    else:
        try:
            article.embedding = article.get_embedding(openaiClient)
            keyword_result = UploadOpenAI(article.inhoud, openaiClient)
            article.add_keywords([keyword['Keyword'] for keyword in json.loads(keyword_result.choices[0].message.content)["metadata"]])
            result_labels, assigned_labels = assign_labels_a_star(article)
            List_result_labels.append({
                "Artikel": article.artikel_id,
                "Artikel_name": article.artikel,
                "List_result_labels": result_labels,
                "correct_labels": article.labels,
                "score_chosen_items": assigned_labels[0]['mean_score']})
            print('----------------------------------------------------------------------------------------------------------------')
            print(f"Article: {article.artikel_id}, Labels: {result_labels}, correct labels: {article.labels}")
        except Exception as e:
            print(f"Article: {article.artikel_id}, {e}")

---------------------------------------------------------------------------

Stage 0, Level: Initial items

Article ID: 18232
Labels: ['Stoffen', 'Aard', 'Chemische stoffen', 'Anorganische verbindingen', 'Niet-metalen', 'fosfor']
Semantic similarity: 1048466.1248048429
Combined score: 1048466.1248048429

Test Article Keywords: ['xxx']
Article Keywords: []
No overlapping keywords.

Article ID: 18232
Labels: ['Stoffen', 'Aard', 'Chemische stoffen', 'Anorganische verbindingen', 'Niet-metalen', 'jood']
Semantic similarity: 762542.5757204451
Combined score: 762542.5757204451

Test Article Keywords: ['xxx']
Article Keywords: []
No overlapping keywords.

Article ID: 13389
Labels: ['Handhaving', 'Bestuurlijke handhaving', 'Administratief beroep', 'Bestuurlijke geldboetes']
Semantic similarity: 762542.5757204451
Combined score: 762542.5757204451

Test Article Keywords: ['xxx']
Article Keywords: []
No overlapping keywords.

Article ID: 13384
Labels: ['Handhaving', 'Toezicht', 'Toezichtsambtenare

In [135]:
import statistics

def calculate_general_confidence_score(median_top1_score, median_topn_score, alpha=0.6, beta=0.4):
    general_confidence = (alpha * median_top1_score) + (beta * median_topn_score)
    return general_confidence


def calculate_label_accuracy_per_level(data):
    # Initialize counters for correct predictions at each label level
    top1_label_level_counts = {}  
    topn_label_level_counts = {}  

    detailed_results = []

    # Initialize lists for score collection
    top1_scores = []
    topn_scores = []

    # Iterate through each item
    for item in data:
        result_labels_list = item['List_result_labels'] 
        correct_labels = item['correct_labels']          
        score_chosen_items = item.get('score_chosen_items', 0.0)  

        max_levels = 3

        # Initialize the level in label_level_counts if not already
        for level in range(max_levels):
            if level not in top1_label_level_counts:
                top1_label_level_counts[level] = {'correct': 0, 'total': 0}
            if level not in topn_label_level_counts:
                topn_label_level_counts[level] = {'correct': 0, 'total': 0}

        # Top-1 Accuracy
        # Use the first classification result
        if result_labels_list:
            top1_result_labels = result_labels_list[0] 
            top1_correct = True 

            # Iterate through each level and compare the labels
            for level in range(max_levels):
                result_label = top1_result_labels[level] if level < len(top1_result_labels) else None
                correct_label = correct_labels[level] if level < len(correct_labels) else None

                # Compare the labels at the current level
                if result_label == correct_label:
                    top1_label_level_counts[level]['correct'] += 1
                else:
                    top1_correct = False  # Labels do not match at this level

                # Count total comparisons made at this level
                top1_label_level_counts[level]['total'] += 1

            # If all labels match up to max_levels, consider the item correct
            if top1_correct:
                top1_scores.append(score_chosen_items)
                item['top1_correct'] = True
            else:
                item['top1_correct'] = False
        else:
            # No result labels returned
            for level in range(max_levels):
                top1_label_level_counts[level]['total'] += 1  
            item['top1_correct'] = False

        # Top-N Accuracy
        # Check if any of the classifications have the correct label at each level
        for level in range(max_levels):
            correct_label = correct_labels[level] if level < len(correct_labels) else None
            match_found_at_level = False

            for result_labels in result_labels_list:
                result_label = result_labels[level] if level < len(result_labels) else None
                if result_label == correct_label:
                    match_found_at_level = True
                    break

            if match_found_at_level:
                topn_label_level_counts[level]['correct'] += 1

            # Count total comparisons made at this level
            topn_label_level_counts[level]['total'] += 1

        # Check for overall correctness in Top-N
        # We consider an item correct if any of the result_labels exactly match correct_labels up to max_levels
        topn_correct = False
        if result_labels_list:
            for result_labels in result_labels_list:
                match_found = True
                for level in range(max_levels):
                    result_label = result_labels[level] if level < len(result_labels) else None
                    correct_label = correct_labels[level] if level < len(correct_labels) else None
                    if result_label != correct_label:
                        match_found = False
                        break
                if match_found:
                    topn_scores.append(score_chosen_items)
                    topn_correct = True
                    item['topn_correct'] = True
                    break
            if not topn_correct:
                item['topn_correct'] = False
        else:
            item['topn_correct'] = False

        # Calculate individual confidence scores
        # Confidence is the score_chosen_items if correct, else 0
        item['confidence_top1'] = score_chosen_items if item['top1_correct'] else 0.0
        item['confidence_topn'] = score_chosen_items if item['topn_correct'] else 0.0

        detailed_results.append({
            'Artikel': item['Artikel'],
            'Artikel_name': item['Artikel_name'],
            'result_labels': result_labels_list,
            'correct_labels': correct_labels,
            'score_chosen_items': score_chosen_items,
            'top1_correct': item['top1_correct'],
            'topn_correct': item['topn_correct'],
            'confidence_top1': item['confidence_top1'],
            'confidence_topn': item['confidence_topn']
        })

    # Calculate Top-1 accuracy per label level
    top1_accuracy_per_label_level = {
        level: (count['correct'] / count['total']) * 100 if count['total'] > 0 else 0
        for level, count in top1_label_level_counts.items()
    }

    # Calculate Top-N accuracy per label level
    topn_accuracy_per_label_level = {
        level: (count['correct'] / count['total']) * 100 if count['total'] > 0 else 0
        for level, count in topn_label_level_counts.items()
    }

    # Calculate median scores
    median_top1_score = statistics.median(top1_scores) if top1_scores else 0.0
    median_topn_score = statistics.median(topn_scores) if topn_scores else 0.0

    return detailed_results, top1_accuracy_per_label_level, topn_accuracy_per_label_level, median_top1_score, median_topn_score


Median confidence score test

In [58]:
# Calculate accuracies and median scores
detailed_results, top1_accuracy, topn_accuracy, median_top1_score, median_topn_score = calculate_label_accuracy_per_level(List_result_labels)

# Print Top-1 Accuracy per Label Level
print("Top-1 Accuracy per Label Level:")
for level, accuracy in top1_accuracy.items():
    print(f"Level {level}: {accuracy:.2f}%")
print(f"Median Top-1 Score: {median_top1_score:.4f}")

# Print Top-N Accuracy per Label Level
print("\nTop-N Accuracy per Label Level:")
for level, accuracy in topn_accuracy.items():
    print(f"Level {level}: {accuracy:.2f}%")
print(f"Median Top-N Score: {median_topn_score:.4f}")

Top-1 Accuracy per Label Level:
Level 0: 95.45%
Level 1: 81.82%
Level 2: 61.36%
Median Top-1 Score: 3.1947

Top-N Accuracy per Label Level:
Level 0: 95.45%
Level 1: 81.82%
Level 2: 77.27%
Median Top-N Score: 2.8621


In [53]:
# Calculate accuracies and median scores
detailed_results, top1_accuracy, topn_accuracy, median_top1_score, median_topn_score = calculate_label_accuracy_per_level(List_result_labels)

# Print Top-1 Accuracy per Label Level
print("Top-1 Accuracy per Label Level:")
for level, accuracy in top1_accuracy.items():
    print(f"Level {level}: {accuracy:.2f}%")
print(f"Median Top-1 Score: {median_top1_score:.4f}")

# Print Top-N Accuracy per Label Level
print("\nTop-N Accuracy per Label Level:")
for level, accuracy in topn_accuracy.items():
    print(f"Level {level}: {accuracy:.2f}%")
print(f"Median Top-N Score: {median_topn_score:.4f}")

Top-1 Accuracy per Label Level:
Level 0: 88.64%
Level 1: 77.27%
Level 2: 52.27%
Median Top-1 Score: 1.8526

Top-N Accuracy per Label Level:
Level 0: 88.64%
Level 1: 77.27%
Level 2: 61.36%
Median Top-N Score: 1.7393


In [387]:
# Calculate accuracies and median scores
detailed_results, top1_accuracy, topn_accuracy, median_top1_score, median_topn_score = calculate_label_accuracy_per_level(List_result_labels)

# Print Top-1 Accuracy per Label Level
print("Top-1 Accuracy per Label Level:")
for level, accuracy in top1_accuracy.items():
    print(f"Level {level}: {accuracy:.2f}%")
print(f"Median Top-1 Score: {median_top1_score:.4f}")

# Print Top-N Accuracy per Label Level
print("\nTop-N Accuracy per Label Level:")
for level, accuracy in topn_accuracy.items():
    print(f"Level {level}: {accuracy:.2f}%")
print(f"Median Top-N Score: {median_topn_score:.4f}")

Top-1 Accuracy per Label Level:
Level 0: 86.86%
Level 1: 75.43%
Level 2: 61.14%
Median Top-1 Score: 1.8367

Top-N Accuracy per Label Level:
Level 0: 86.86%
Level 1: 75.43%
Level 2: 62.29%
Median Top-N Score: 1.8367


Mean confidence score test

In [382]:
# Calculate accuracies and mean scores
detailed_results, top1_accuracy, topn_accuracy, mean_top1_score, mean_topn_score = calculate_label_accuracy_per_level(List_result_labels)

# Print Top-1 Accuracy per Label Level
print("Top-1 Accuracy per Label Level:")
for level, accuracy in top1_accuracy.items():
    print(f"Level {level}: {accuracy:.2f}%")
print(f"Mean Top-1 Score: {mean_top1_score:.4f}")

# Print Top-N Accuracy per Label Level
print("\nTop-N Accuracy per Label Level:")
for level, accuracy in topn_accuracy.items():
    print(f"Level {level}: {accuracy:.2f}%")
print(f"Mean Top-N Score: {mean_topn_score:.4f}")

Top-1 Accuracy per Label Level:
Level 0: 87.23%
Level 1: 82.98%
Level 2: 74.47%
Mean Top-1 Score: 2.5765

Top-N Accuracy per Label Level:
Level 0: 87.23%
Level 1: 82.98%
Level 2: 74.47%
Mean Top-N Score: 2.5765


In [375]:
# Calculate accuracies and mean scores
detailed_results, top1_accuracy, topn_accuracy, mean_top1_score, mean_topn_score = calculate_label_accuracy_per_level(List_result_labels)

# Print Top-1 Accuracy per Label Level
print("Top-1 Accuracy per Label Level:")
for level, accuracy in top1_accuracy.items():
    print(f"Level {level}: {accuracy:.2f}%")
print(f"Mean Top-1 Score: {mean_top1_score:.4f}")

# Print Top-N Accuracy per Label Level
print("\nTop-N Accuracy per Label Level:")
for level, accuracy in topn_accuracy.items():
    print(f"Level {level}: {accuracy:.2f}%")
print(f"Mean Top-N Score: {mean_topn_score:.4f}")

Top-1 Accuracy per Label Level:
Level 0: 95.83%
Level 1: 87.50%
Level 2: 79.17%
Mean Top-1 Score: 2.4144

Top-N Accuracy per Label Level:
Level 0: 95.83%
Level 1: 87.50%
Level 2: 79.17%
Mean Top-N Score: 2.4144
